In [1]:
# Force kernel restart and reimport
import sys
# Clear cached modules to force reimport
for mod in list(sys.modules.keys()):
    if 'base' in mod or 'data_loader' in mod or 'ModelWrapper' in mod:
        del sys.modules[mod]

In [2]:
# Setup paths and imports
from pathlib import Path
import os
import sys
from typing import Optional
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from importlib import reload

# Ensure project roots are on PYTHONPATH
PROJECT_ROOT = Path().resolve()
if not (PROJECT_ROOT / "SubModules").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add src directory to path for base module
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Add ADBench to path
ADBENCH_ROOT = PROJECT_ROOT / "SubModules" / "ADBench"
if str(ADBENCH_ROOT) not in sys.path:
    sys.path.append(str(ADBENCH_ROOT))

# Import unified interfaces, wrappers, and data loader
import base
base = reload(base)
from base import Model, RecurrentModel, Data, CoLearning, CoLearner, SingleModel
import baselines.adbench.ModelWrapperADBench as mw
mw = reload(mw)
from baselines.adbench.ModelWrapperADBench import PYOD_AVAILABLE
from baselines.adbench.data_loader import load_adbench_classical, load_data

print("✓ All imports successful")
print("\nIMPORTANT: If you see PyOD import errors, run:")
print("  pip install scikit-learn==1.0.2 pyod==1.0.9 --no-cache-dir")

2026-01-19 10:42:14.212498: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-19 10:42:14.212826: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 10:42:14.261577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 10:42:15.501824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

✓ All imports successful

IMPORTANT: If you see PyOD import errors, run:
  pip install scikit-learn==1.0.2 pyod==1.0.9 --no-cache-dir


# MVP Notebook: Testing Model Wrappers

## Setup Instructions

### PyOD Binary Compatibility Error Fix

pip uninstall scikit-learn pyod -y
pip install --no-cache-dir scikit-learn==1.0.2
pip install --no-cache-dir pyod==1.0.9

```
- All wrappers now live in `baselines/adbench/ModelWrapperADBench.py`

In [3]:
# Load ADBench Classical dataset using new Data class

DATASET_PATH = PROJECT_ROOT / "SubModules/ADBench/adbench/datasets/Classical/2_annthyroid.npz"

# Load as ClassicalADBenchData (replaces dict)
data = load_data(DATASET_PATH)

print(f"✓ Loaded dataset: {data}")
print(f"  Training samples: {data.n_train}, features: {data.X_train.shape[1]}")
print(f"  Test samples: {data.n_test}")
print(f"\nAccess patterns:")
print(f"  - Object API: data.X_train, data.y_train, data.X_test, data.y_test")
print(f"  - Dict API (backward-compat): data['X_train'], data.get('y_train')")


✓ Loaded dataset: ClassicalADBenchData(dataset=2_annthyroid.npz, n_train=5760, n_test=1440, n_features=6)
  Training samples: 5760, features: 6
  Test samples: 1440

Access patterns:
  - Object API: data.X_train, data.y_train, data.X_test, data.y_test
  - Dict API (backward-compat): data['X_train'], data.get('y_train')


In [4]:
# Test wrappers with new Data class

print("=" * 60)
print("MVP: Testing Model Wrappers with ClassicalADBenchData")
print("=" * 60)

registry = mw.get_model_detector_dict()
results_summary = {}

# 1. Test PReNet
print("\n" + "=" * 60)
print("1. PReNet with Epoch-Level Training")
print("=" * 60)

prenet_cls = registry["prenet"]
train_config = {"total_epochs": 5, "batch_size": 256}
prenet = prenet_cls(train_config=train_config, model_config={}, data=data)

print("Training PReNet epoch-by-epoch:")
for epoch in range(5):
    prenet.train(epoch)
    scores = prenet.predict_scores()
    auc_score = roc_auc_score(data.y_test, scores)
    print(f"  Epoch {epoch}: ROC-AUC = {auc_score:.4f}")

results_summary["PReNet"] = auc_score

# 2. Test DeepSAD
print("\n" + "=" * 60)
print("2. DeepSAD with Autoencoder Pretraining")
print("=" * 60)

deepsad_cls = registry["deepsad"]
train_config = {"total_epochs": 10, "pretrain": True, "ae_epochs": 20, "batch_size": 128}
deep = deepsad_cls(train_config=train_config, model_config={}, data=data)

print("\nTraining DeepSAD epoch-by-epoch:")
for epoch in range(10):
    deep.train(epoch)
    if epoch % 2 == 0:
        scores = deep.predict_scores()
        auc_score = roc_auc_score(data.y_test, scores)
        print(f"  Epoch {epoch}: ROC-AUC = {auc_score:.4f}")

scores = deep.predict_scores()
auc_score = roc_auc_score(data.y_test, scores)
results_summary["DeepSAD"] = auc_score

# 3. Test DevNet
print("\n" + "=" * 60)
print("3. DevNet with Single-Epoch Steps")
print("=" * 60)

try:
    devnet_cls = registry["devnet"]
    train_config = {"total_epochs": 3, "batch_size": 256, "nb_batch": 5, "network_depth": 2}
    devnet = devnet_cls(train_config=train_config, model_config={}, data=data)

    print("\nTraining DevNet epoch-by-epoch:")
    for epoch in range(train_config["total_epochs"]):
        devnet.train(epoch)
        scores = devnet.predict_scores()
        auc_score_dev = roc_auc_score(data.y_test, scores)
        print(f"  Epoch {epoch}: ROC-AUC = {auc_score_dev:.4f}")

    results_summary["DevNet"] = auc_score_dev
except Exception as e:
    print(f"⚠ DevNet test failed: {type(e).__name__}: {e}")

# 4. Test XGBOD if available
if PYOD_AVAILABLE:
    print("\n" + "=" * 60)
    print("4. XGBOD with Refit Strategy")
    print("=" * 60)

    xgbod_cls = registry["xgbod"]
    train_config = {"total_epochs": 3}
    xgbod = xgbod_cls(train_config=train_config, model_config={}, data=data)

    print("\nTraining XGBOD (refits each epoch):")
    for epoch in range(1):
        xgbod.train(epoch)
        scores = xgbod.predict_scores()
        auc_score_xgb = roc_auc_score(data.y_test, scores)
        print(f"  Epoch {epoch}: ROC-AUC = {auc_score_xgb:.4f}")

    results_summary["XGBOD"] = auc_score_xgb
else:
    print("\n⚠ XGBOD skipped - PyOD not available")

# Final Summary
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
for model_name, auc in results_summary.items():
    print(f"{model_name:15s}: ROC-AUC = {auc:.4f}")
print("=" * 60)

MVP: Testing Model Wrappers with ClassicalADBenchData

1. PReNet with Epoch-Level Training
Training PReNet epoch-by-epoch:
  Epoch 0: ROC-AUC = 0.5756
  Epoch 1: ROC-AUC = 0.6569
  Epoch 2: ROC-AUC = 0.6876
  Epoch 3: ROC-AUC = 0.7058
  Epoch 4: ROC-AUC = 0.7180

2. DeepSAD with Autoencoder Pretraining

Training DeepSAD epoch-by-epoch:
  Epoch 0: ROC-AUC = 0.8205
  Epoch 2: ROC-AUC = 0.9571
  Epoch 4: ROC-AUC = 0.9805
  Epoch 6: ROC-AUC = 0.9847
  Epoch 8: ROC-AUC = 0.9869

3. DevNet with Single-Epoch Steps

Training DevNet epoch-by-epoch:


E0000 00:00:1768815748.283190   36720 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1768815748.302709   36720 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 965us/step
  Epoch 0: ROC-AUC = 0.6088
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step
  Epoch 1: ROC-AUC = 0.6390
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
  Epoch 2: ROC-AUC = 0.6600

4. XGBOD with Refit Strategy

Training XGBOD (refits each epoch):
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:42:45] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Epoch 0: ROC-AUC = 0.9942

FINAL RESULTS SUMMARY
PReNet         : ROC-AUC = 0.7180
DeepSAD        : ROC-AUC = 0.9869
DevNet         : ROC-AUC = 0.6600
XGBOD          : ROC-AUC = 0.9942


# Test Collaborative Learning (CoLearner)

In [5]:
# Force reload of base module to get latest changes
from importlib import reload
import base
base = reload(base)
from base import CoLearner, SingleModel, SimpleStrategy


In [6]:

# Initialize models for collaborative learning
# Use simpler models to avoid long training times
models_for_colearing = [
    prenet,      # PReNet
    xgbod,       # XGBOD (fast, tree-based)
]

# Create simple strategy for convergence control
strategy = SimpleStrategy(max_chapters=50, patience=5, patience_threshold=0.001)

# Initialize CoLearner
print("Initializing CoLearner with 2 models (PReNet, XGBOD)...")
co_learner = CoLearner(
    models=models_for_colearing,
    data=data,
    strategy=strategy,
    warmup_epochs=1,      # Short warmup for demo
    max_chapters=1         # Short training for demo
)

# Run collaborative training
print("\nStarting collaborative training loop...")
history = co_learner.cotrain(recurrent_model=None, eval_interval=1)

# Print final results
print("\n" + "="*60)
print("COLLABORATIVE LEARNING RESULTS")
print("="*60)
final_metrics = history["chapters"][-1] if history["chapters"] else {}
for model_name, auc in final_metrics.items():
    print(f"{model_name:15s}: ROC-AUC = {auc:.4f}")

Initializing CoLearner with 2 models (PReNet, XGBOD)...

Starting collaborative training loop...

[Warmup] Training 2 models for 1 epochs (no exchange)...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:43:02] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Epoch 1/1

[Collaborative] Running up to 1 chapters...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:43:30] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 1: model_0=0.7316, model_1=0.8067, ensemble=0.7465

[Done] Collaborative training complete

COLLABORATIVE LEARNING RESULTS
model_0        : ROC-AUC = 0.7316
model_1        : ROC-AUC = 0.8067
ensemble       : ROC-AUC = 0.7465


# Detailed CoLearner Testing with Index Selection

In this section, we test the refactored CoLearner with:
- Real models (PReNet and XGBOD)
- Real data (ClassicalADBenchData)
- Index-based pseudo-label exchange via `get_anomaly_indexes()` and `get_normal_indexes()`


In [7]:
# TEST 1: Index Selection Functions

print("=" * 70)
print("TEST 1: Index Selection with get_anomaly_indexes and get_normal_indexes")
print("=" * 70)

# Generate dummy scores for demonstration
dummy_scores = np.concatenate([
    np.random.uniform(0.0, 0.4, 100),  # Normals
    np.random.uniform(0.6, 1.0, 100),  # Anomalies
])

print(f"\nDummy scores created: {len(dummy_scores)} samples")
print(f"  Score range: [{dummy_scores.min():.4f}, {dummy_scores.max():.4f}]")

# Create a test CoLearner to use index functions
test_strategy = SimpleStrategy(max_chapters=1, patience=5)
test_colearner = CoLearner(
    models=[prenet, xgbod],
    data=data,
    strategy=test_strategy,
    warmup_epochs=0,
    max_chapters=1,
    anomaly_threshold=0.5
)

anomaly_indexes = test_colearner.get_anomaly_indexes(dummy_scores)
normal_indexes = test_colearner.get_normal_indexes(dummy_scores)

print(f"\nAnomaly indexes (scores > 0.5): {len(anomaly_indexes)} samples")
print(f"  First 10 indices: {anomaly_indexes[:10]}")
print(f"  Score range: [{dummy_scores[anomaly_indexes].min():.4f}, {dummy_scores[anomaly_indexes].max():.4f}]")

print(f"\nNormal indexes (scores <= 0.5): {len(normal_indexes)} samples")
print(f"  First 10 indices: {normal_indexes[:10]}")
print(f"  Score range: [{dummy_scores[normal_indexes].min():.4f}, {dummy_scores[normal_indexes].max():.4f}]")

print(f"\n✓ Index selection working correctly!")
print(f"  Total indices: {len(anomaly_indexes) + len(normal_indexes)} = {len(dummy_scores)}")

TEST 1: Index Selection with get_anomaly_indexes and get_normal_indexes

Dummy scores created: 200 samples
  Score range: [0.0036, 0.9988]

Anomaly indexes (scores > 0.5): 100 samples
  First 10 indices: [100 101 102 103 104 105 106 107 108 109]
  Score range: [0.6002, 0.9988]

Normal indexes (scores <= 0.5): 100 samples
  First 10 indices: [0 1 2 3 4 5 6 7 8 9]
  Score range: [0.0036, 0.3979]

✓ Index selection working correctly!
  Total indices: 200 = 200


In [8]:
# TEST 2: Single Exchange Round (Manual)

print("\n" + "=" * 70)
print("TEST 2: Single Exchange Round - Manual send_anomalies/send_normals")
print("=" * 70)

# Create a fresh data object for this test
data_exchange = load_data(DATASET_PATH)
print(f"\n✓ Loaded fresh data: {data_exchange.n_train} training samples")

# Get predictions from PReNet on training set
print("\n1. Getting predictions from PReNet on training set...")
prenet_scores = prenet.predict_scores()
print(f"   Score range: [{prenet_scores.min():.4f}, {prenet_scores.max():.4f}]")
print(f"   Mean score: {np.mean(prenet_scores):.4f}")

# Get predictions from XGBOD on training set
print("\n2. Getting predictions from XGBOD on training set...")
xgbod_scores = xgbod.predict_scores()
print(f"   Score range: [{xgbod_scores.min():.4f}, {xgbod_scores.max():.4f}]")
print(f"   Mean score: {np.mean(xgbod_scores):.4f}")

# Create CoLearner
print("\n3. Creating CoLearner instance...")
strategy = SimpleStrategy(max_chapters=1, patience=5)
colearner = CoLearner(
    models=[prenet, xgbod],
    data=data_exchange,
    strategy=strategy,
    warmup_epochs=0,
    max_chapters=1,
    anomaly_threshold=0.5
)
print("   ✓ CoLearner created")

# Manual exchange: PReNet sends to XGBOD
print("\n4. PReNet sends its anomalies and normals to XGBOD...")

# Get index selections from PReNet scores
prenet_anomaly_idx = colearner.get_anomaly_indexes(prenet_scores)
prenet_normal_idx = colearner.get_normal_indexes(prenet_scores)

print(f"   PReNet detected: {len(prenet_anomaly_idx)} anomalies, {len(prenet_normal_idx)} normals")

# Send to XGBOD
colearner.send_anomalies(model_idx=0, anomaly_indexes=prenet_anomaly_idx, scores=prenet_scores)
colearner.send_normals(model_idx=0, normal_indexes=prenet_normal_idx, scores=prenet_scores)

# Check XGBOD's received pseudo-labels
xgbod_pseudo = data_exchange.get_pseudo_labels("model_1")
xgbod_conf = data_exchange.pseudo_label_confidence["model_1"]

print(f"\n   XGBOD received:")
print(f"     - Pseudo-labels from PReNet: {len(np.unique(xgbod_pseudo))} unique values")
print(f"     - Mean confidence: {np.mean(xgbod_conf):.4f}")
print(f"     - Anomalies marked (label=1): {np.sum(xgbod_pseudo == 1)}")
print(f"     - Normals marked (label=0): {np.sum(xgbod_pseudo == 0)}")

# Manual exchange: XGBOD sends to PReNet
print("\n5. XGBOD sends its anomalies and normals to PReNet...")

xgbod_anomaly_idx = colearner.get_anomaly_indexes(xgbod_scores)
xgbod_normal_idx = colearner.get_normal_indexes(xgbod_scores)

print(f"   XGBOD detected: {len(xgbod_anomaly_idx)} anomalies, {len(xgbod_normal_idx)} normals")

colearner.send_anomalies(model_idx=1, anomaly_indexes=xgbod_anomaly_idx, scores=xgbod_scores)
colearner.send_normals(model_idx=1, normal_indexes=xgbod_normal_idx, scores=xgbod_scores)

# Check PReNet's received pseudo-labels
prenet_pseudo = data_exchange.get_pseudo_labels("model_0")
prenet_conf = data_exchange.pseudo_label_confidence["model_0"]

print(f"\n   PReNet received:")
print(f"     - Pseudo-labels from XGBOD: {len(np.unique(prenet_pseudo))} unique values")
print(f"     - Mean confidence: {np.mean(prenet_conf):.4f}")
print(f"     - Anomalies marked (label=1): {np.sum(prenet_pseudo == 1)}")
print(f"     - Normals marked (label=0): {np.sum(prenet_pseudo == 0)}")

print(f"\n✓ Manual exchange round completed successfully!")


TEST 2: Single Exchange Round - Manual send_anomalies/send_normals

✓ Loaded fresh data: 5760 training samples

1. Getting predictions from PReNet on training set...
   Score range: [0.0000, 1.0000]
   Mean score: 0.3130

2. Getting predictions from XGBOD on training set...
   Score range: [0.0000, 1.0000]
   Mean score: 0.0568

3. Creating CoLearner instance...
   ✓ CoLearner created

4. PReNet sends its anomalies and normals to XGBOD...
   PReNet detected: 16 anomalies, 1424 normals

   XGBOD received:
     - Pseudo-labels from PReNet: 2 unique values
     - Mean confidence: 0.0472
     - Anomalies marked (label=1): 427
     - Normals marked (label=0): 5333

5. XGBOD sends its anomalies and normals to PReNet...
   XGBOD detected: 74 anomalies, 1366 normals

   PReNet received:
     - Pseudo-labels from XGBOD: 2 unique values
     - Mean confidence: 0.1196
     - Anomalies marked (label=1): 427
     - Normals marked (label=0): 5333

✓ Manual exchange round completed successfully!


In [9]:
# TEST 3: Full Collaborative Training with exchange()

print("\n" + "=" * 70)
print("TEST 3: Full Collaborative Training Loop with exchange()")
print("=" * 70)

# Create fresh data for clean test
data_collearn = load_data(DATASET_PATH)

# Use existing trained models
models_list = [prenet, xgbod]

strategy = SimpleStrategy(max_chapters=50, patience=3, patience_threshold=0.001)

colearner_full = CoLearner(
    models=models_list,
    data=data_collearn,
    strategy=strategy,
    warmup_epochs=1,      # Minimal warmup
    max_chapters=3,       # Just 3 chapters for demo
    anomaly_threshold=0.5
)

print(f"\n✓ Created CoLearner with {len(models_list)} models")
print(f"  Warmup epochs: {colearner_full.warmup_epochs}")
print(f"  Max chapters: {colearner_full.max_chapters}")
print(f"  Anomaly threshold: {colearner_full.anomaly_threshold}")

# Run full collaborative training
print("\nRunning collaborative training...")
history = colearner_full.cotrain(recurrent_model=None, eval_interval=1)

# Analyze results
print("\n" + "=" * 70)
print("RESULTS: Collaborative Training History")
print("=" * 70)

if history["warmup"]:
    print("\nWarmup Phase Metrics:")
    for i, metrics in enumerate(history["warmup"]):
        print(f"  Epoch {i}: {metrics}")

if history["chapters"]:
    print("\nCollaborative Chapters Metrics:")
    for i, metrics in enumerate(history["chapters"]):
        print(f"  Chapter {i}:")
        for model, auc in metrics.items():
            print(f"    {model:15s}: {auc:.4f}")

# Get final ensemble AUC
if history["chapters"]:
    final_ensemble_auc = history["chapters"][-1].get("ensemble", 0.0)
    print(f"\n✓ Final Ensemble ROC-AUC: {final_ensemble_auc:.4f}")


TEST 3: Full Collaborative Training Loop with exchange()

✓ Created CoLearner with 2 models
  Warmup epochs: 1
  Max chapters: 3
  Anomaly threshold: 0.5

Running collaborative training...

[Warmup] Training 2 models for 1 epochs (no exchange)...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:43:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Epoch 1/1

[Collaborative] Running up to 3 chapters...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:44:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 1: model_0=0.7402, model_1=0.8107, ensemble=0.7520
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:44:50] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 2: model_0=0.7297, model_1=0.8130, ensemble=0.7495
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:45:19] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 3: model_0=0.7244, model_1=0.7925, ensemble=0.7381

[Done] Collaborative training complete

RESULTS: Collaborative Training History

Collaborative Chapters Metrics:
  Chapter 0:
    model_0        : 0.7402
    model_1        : 0.8107
    ensemble       : 0.7520
  Chapter 1:
    model_0        : 0.7297
    model_1        : 0.8130
    ensemble       : 0.7495
  Chapter 2:
    model_0        : 0.7244
    model_1        : 0.7925
    ensemble       : 0.7381

✓ Final Ensemble ROC-AUC: 0.7381


In [10]:
# TEST 4: Pseudo-Label Propagation Analysis

print("\n" + "=" * 70)
print("TEST 4: Analyzing Pseudo-Label Propagation")
print("=" * 70)

# After exchange, examine how labels propagated
print("\nPseudo-Label Distribution After Exchange:")
print("-" * 70)

# Get all pseudo-labels from the full collaborative training
for model_idx in range(len(models_list)):
    model_name = f"model_{model_idx}"
    if model_name in data_collearn.pseudo_labels_by_model:
        labels = data_collearn.pseudo_labels_by_model[model_name]
        confidence = data_collearn.pseudo_label_confidence[model_name]
        
        print(f"\n{model_name}:")
        print(f"  Pseudo-labels: {np.sum(labels == 0)} normals, {np.sum(labels == 1)} anomalies")
        print(f"  Confidence: mean={np.mean(confidence):.4f}, std={np.std(confidence):.4f}")
        print(f"  Confidence range: [{np.min(confidence):.4f}, {np.max(confidence):.4f}]")

# Compare with original labels
print(f"\nOriginal Training Labels:")
print(f"  Class 0 (normal): {np.sum(data_collearn.y_train == 0)}")
print(f"  Class 1 (anomaly): {np.sum(data_collearn.y_train == 1)}")

print("\n✓ Pseudo-label analysis complete!")


TEST 4: Analyzing Pseudo-Label Propagation

Pseudo-Label Distribution After Exchange:
----------------------------------------------------------------------

Original Training Labels:
  Class 0 (normal): 5333
  Class 1 (anomaly): 427

✓ Pseudo-label analysis complete!


In [11]:
# TEST 5: Comparison - Single Model vs Collaborative

print("\n" + "=" * 70)
print("TEST 5: Single Model Baseline vs Collaborative Learning")
print("=" * 70)

# Load fresh data for single model test
data_single = load_data(DATASET_PATH)

print("\n1. Training Single Model (PReNet only, no collaboration):")
print("-" * 70)

single_learner = SingleModel(
    model=prenet,
    data=data_single,
    strategy=None,
    warmup_epochs=0
)

# Train single model
single_learner.model.fit()
single_scores = single_learner.model.predict_scores()
single_auc = roc_auc_score(data_single.y_test, single_scores)
print(f"Single PReNet ROC-AUC: {single_auc:.4f}")

# Compare with collaborative learning results
print("\n2. Comparing with Collaborative Learning Results:")
print("-" * 70)

if history["chapters"]:
    collab_prenet_auc = history["chapters"][-1].get("model_0", 0.0)
    collab_xgbod_auc = history["chapters"][-1].get("model_1", 0.0)
    collab_ensemble_auc = history["chapters"][-1].get("ensemble", 0.0)
    
    print(f"PReNet (single):        {single_auc:.4f}")
    print(f"PReNet (collaborative): {collab_prenet_auc:.4f}")
    print(f"XGBOD (collaborative):  {collab_xgbod_auc:.4f}")
    print(f"Ensemble:               {collab_ensemble_auc:.4f}")
    
    improvement = collab_prenet_auc - single_auc
    pct_improvement = (improvement / single_auc) * 100 if single_auc > 0 else 0
    
    print(f"\nImprovement from Collaboration:")
    print(f"  Absolute: {improvement:+.4f}")
    print(f"  Relative: {pct_improvement:+.2f}%")


TEST 5: Single Model Baseline vs Collaborative Learning

1. Training Single Model (PReNet only, no collaboration):
----------------------------------------------------------------------
Single PReNet ROC-AUC: 0.7187

2. Comparing with Collaborative Learning Results:
----------------------------------------------------------------------
PReNet (single):        0.7187
PReNet (collaborative): 0.7244
XGBOD (collaborative):  0.7925
Ensemble:               0.7381

Improvement from Collaboration:
  Absolute: +0.0057
  Relative: +0.79%


# Summary of CoLearner MVP Tests

## What Was Tested

### TEST 1: Index Selection Functions
- **get_anomaly_indexes()**: Identifies samples where scores > threshold
- **get_normal_indexes()**: Identifies samples where scores ≤ threshold
- Verifies index selection returns correct counts and score ranges

### TEST 2: Manual Exchange Round
- **send_anomalies()**: Sends detected anomalies to other models
- **send_normals()**: Sends detected normals to other models
- Bi-directional pseudo-label exchange between PReNet and XGBOD
- Verifies confidence scores are properly calculated and stored

### TEST 3: Full Collaborative Training
- **exchange()**: Automated pseudo-label exchange between all models
- **cotrain()**: Complete training loop with metrics evaluation
- Convergence checking with SimpleStrategy
- Ensemble score computation per chapter

### TEST 4: Pseudo-Label Propagation Analysis
- Tracks how labels propagate between models
- Analyzes confidence distribution after exchange
- Compares generated labels with original training labels

### TEST 5: Single Model vs Collaborative
- Compares single model training (baseline) against collaborative learning
- Measures absolute and relative improvement
- Shows ensemble benefits

## Key Implementation Features

✓ **Index-based exchange**: Efficient pseudo-label propagation without full array copies
✓ **Confidence-weighted labels**: Ensures high-quality label sharing
✓ **Bi-directional learning**: Models learn both anomalies and normals from each other
✓ **Convergence strategy**: Stops when ensemble AUC plateaus
✓ **Ensemble prediction**: Combines predictions from multiple models

## Next Steps

- Experiment with different threshold values
- Add more models to the collaborative ensemble
- Test with different datasets (GADBench, PyGOD)
- Implement recurrent model training on aggregated embeddings
- Track individual model improvements during collaboration

# Semi-Supervised Learning Tests

Testing the new `preserve_labeled` functionality to enable semi-supervised collaborative learning.

In [ ]:
# TEST 6: Mode A - Overwrite All Labels (preserve_labeled=False)
print("="*70)
print("TEST 6: Mode A - Overwrite All Labels (preserve_labeled=False)")
print("="*70)

# Load data without preserving labels
data_mode_a = load_data(DATASET_PATH, preserve_labeled=False)
data_mode_a.generate_semisupervised_split(labeled_ratio=0.1, stratified=True)
print(f"\n✓ Data: {data_mode_a.n_train} training samples")
print(f"  - Labeled: {len(data_mode_a.labeled_indexes)} ({data_mode_a.labeled_ratio:.1%})")
print(f"  - Unlabeled: {len(data_mode_a.unlabeled_indexes)}")

# Reuse trained models with new data
prenet.data = data_mode_a
xgbod.data = data_mode_a

# Run collaborative training
strategy_a = SimpleStrategy(max_chapters=3, patience=3)
colearner_a = CoLearner(
    models=[prenet, xgbod],
    data=data_mode_a,
    strategy=strategy_a,
    warmup_epochs=1,
    max_chapters=3,
    anomaly_threshold=0.5
)

print("\nRunning collaborative training (Mode A: overwrite all labels)...")
history_a = colearner_a.cotrain(eval_interval=1)

final_metrics_a = history_a['chapters'][-1]
labeled_preserved = np.array_equal(
    prenet._pseudo_labels[data_mode_a.labeled_indexes],
    data_mode_a.y_train_original[data_mode_a.labeled_indexes]
)

print(f"\n✓ Results: Ensemble={final_metrics_a['ensemble']:.4f}, PReNet={final_metrics_a['model_0']:.4f}, XGBOD={final_metrics_a['model_1']:.4f}")
print(f"  Labeled samples preserved: {labeled_preserved} (Expected: False)")
print("="*70)

TEST 6: Mode A - Overwrite All Labels (preserve_labeled=False)

✓ Data: 5760 training samples
  - Labeled: 576 (10.0%)
  - Unlabeled: 5184

Running collaborative training (Mode A: overwrite all labels)...

[Warmup] Training 2 models for 1 epochs (no exchange)...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(


In [ ]:
# TEST 7: Mode B - Preserve Labeled Indexes (preserve_labeled=True)
print("="*70)
print("TEST 7: Mode B - Preserve Labeled Indexes (preserve_labeled=True)")
print("="*70)

# Load data WITH preserving labels
data_mode_b = load_data(DATASET_PATH, preserve_labeled=True)
data_mode_b.generate_semisupervised_split(labeled_ratio=0.1, stratified=True)
print(f"\n✓ Data: {data_mode_b.n_train} training samples")
print(f"  - Labeled: {len(data_mode_b.labeled_indexes)} ({data_mode_b.labeled_ratio:.1%})")
print(f"  - Unlabeled: {len(data_mode_b.unlabeled_indexes)}")

# Reuse trained models with new data
prenet.data = data_mode_b
xgbod.data = data_mode_b

# Run collaborative training
strategy_b = SimpleStrategy(max_chapters=3, patience=3)
colearner_b = CoLearner(
    models=[prenet, xgbod],
    data=data_mode_b,
    strategy=strategy_b,
    warmup_epochs=1,
    max_chapters=3,
    anomaly_threshold=0.5
)

print("\nRunning collaborative training (Mode B: preserve labeled labels)...")
history_b = colearner_b.cotrain(eval_interval=1)

final_metrics_b = history_b['chapters'][-1]
labeled_preserved = np.array_equal(
    prenet._pseudo_labels[data_mode_b.labeled_indexes],
    data_mode_b.y_train_original[data_mode_b.labeled_indexes]
)

print(f"\n✓ Results: Ensemble={final_metrics_b['ensemble']:.4f}, PReNet={final_metrics_b['model_0']:.4f}, XGBOD={final_metrics_b['model_1']:.4f}")
print(f"  Labeled samples preserved: {labeled_preserved} (Expected: True)")
print("="*70)

TEST 7: Mode B - Preserve Labeled Indexes (preserve_labeled=True)
✓ Data loaded: preserve_labeled=True
  Train samples: 5760
  Test samples: 1440

✓ Semi-supervised split created:
  Labeled: 576 samples (10.0%)
  Unlabeled: 5184 samples

✓ Starting collaborative training (Mode B)...

[Warmup] Training 2 models for 1 epochs (no exchange)...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [08:25:12] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Epoch 1/1

[Collaborative] Running up to 3 chapters...
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [08:25:40] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 1: model_0=0.7640, model_1=0.9942, ensemble=0.9912
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [08:26:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 2: model_0=0.7650, model_1=0.9942, ensemble=0.9912
best param: None


/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/pyod/models/base.py:431: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/home/peppino58/miniconda3/envs/cobench/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [08:27:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "silent" } are not used.

  warnings.warn(smsg, UserWarning)


  Chapter 3: model_0=0.7659, model_1=0.9942, ensemble=0.9912

[Done] Collaborative training complete

✓ Mode B Results:
  Ensemble AUC: 0.9912
  PReNet AUC: 0.7659
  XGBOD AUC: 0.9942
  Labeled samples preserved: False (Expected: True)



In [ ]:
# TEST 8: Mode Comparison
print("="*70)
print("TEST 8: Semi-Supervised Mode Comparison")
print("="*70)

import pandas as pd

# Create comparison table
comparison = {
    'Mode': ['A: Overwrite All', 'B: Preserve Labeled'],
    'Ensemble AUC': [
        history_a['chapters'][-1]['ensemble'],
        history_b['chapters'][-1]['ensemble']
    ],
    'PReNet AUC': [
        history_a['chapters'][-1]['model_0'],
        history_b['chapters'][-1]['model_0']
    ],
    'XGBOD AUC': [
        history_a['chapters'][-1]['model_1'],
        history_b['chapters'][-1]['model_1']
    ]
}

df_comparison = pd.DataFrame(comparison)
print("\n✓ Comparison Results:")
print(df_comparison.to_string(index=False))

ensemble_diff = history_b['chapters'][-1]['ensemble'] - history_a['chapters'][-1]['ensemble']
print(f"\nEnsemble AUC Delta (Mode B - Mode A): {ensemble_diff:+.4f}")

if ensemble_diff > 0:
    print("→ Preserving labeled samples IMPROVED performance")
elif ensemble_diff < 0:
    print("→ Overwriting all labels performed BETTER")
else:
    print("→ No significant difference between modes")

print("="*70)

TEST 8: Semi-Supervised Mode Comparison

✓ Results Comparison:
              Mode  preserve_labeled  Ensemble AUC  PReNet AUC  XGBOD AUC
Mode A (Overwrite)             False      0.991173    0.762057   0.994188
 Mode B (Preserve)              True      0.991180    0.765864   0.994188

Δ Ensemble AUC (Mode B - Mode A): +0.0000
→ Preserving labeled samples IMPROVED performance

